# Práctica P1: El Modelo IS-LM Dinámico en Tiempo Continuo
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P1-v1.0
*   **Capítulo de Referencia**: Capítulo 2, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Analizar la respuesta dinámica y la transición temporal de una economía IS-LM cerrada ante shocks de políticas macroeconómicas (monetarias y fiscales) bajo un ajuste gradual de la producción e inflación por curva de Phillips.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el mecanismo de transmisión dinámica de un shock monetario y fiscal en un modelo de equilibrio general simple de corto-medio plazo.
2.  **Visualizar** el principio de *neutralidad del dinero* a largo plazo y la no neutralidad a corto plazo.
3.  **Aprender** a integrar sistemas de ecuaciones diferenciales ordinarias (ODEs) en Python utilizando `scipy.integrate.solve_ivp`.
4.  **Evaluar** críticamente los resultados simulados comparándolos contra el "oráculo" analítico y los apéndices numéricos del libro.


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

> *   **📋 Antes de empezar**, consulta ' (en esta misma carpeta): objetivos, tiempo estimado y conocimientos previos de esta práctica.
> 
### 🕹️ GUÍA RÁPIDA DE INICIO - IS-LM Dinámico
*   **¿Qué estamos haciendo aquí?** Simulamos una economía cerrada. El tipo de interés y la producción se ajustan según la oferta y demanda de bienes y dinero.
*   **Controles clave para cambiar:** 
    *   `m0`: La oferta de dinero (política monetaria).
    *   `beta0`: El gasto del gobierno (política fiscal).
*   **El Shock:** Si subes `m0` (imprimir dinero), verás que a corto plazo el PIB sube y los intereses bajan. Pero a largo plazo, el PIB vuelve a su sitio y solo suben los precios (neutralidad del dinero).
*   **¡Prueba esto!** Usa los deslizadores interactivos al final del cuaderno para arrastrar y ver la animación del shock en vivo.


In [ ]:
%%capture
# "%%capture" es una "magia" de Jupyter: ejecuta todo lo que hay debajo pero
# no imprime nada en pantalla (ni salidas normales ni errores leves). Útil
# para celdas de instalación que generan mucho texto de registro.
# import sys trae el módulo del sistema operativo. La comprobación
# 'google.colab' in sys.modules mira si el módulo de Colab está cargado en
# memoria: solo será True si este notebook se está ejecutando en Google
# Colab. En tu entorno local (venv), esta celda no hará nada.
import sys
if 'google.colab' in sys.modules:
    !pip install numpy scipy matplotlib ipywidgets

In [ ]:
# "import" trae a este cuaderno código escrito en otro sitio, para no tener
# que reescribirlo. "import X as np" trae toda la librería X y le da un
# alias corto (np) para no teclear el nombre completo cada vez.
# "from X import Y" es más selectivo: trae solo la función/clase Y de X.
# sys.path.append('../../src') añade la carpeta src/ del proyecto al PATH de
# Python para que encuentre el paquete macroaicomp aunque estemos dos niveles
# más abajo (en practicas/01-is-lm-dinamico/).

# Librerías de terceros (instaladas en el entorno con pip)
import numpy as np                      # cálculo numérico: vectores, matrices, álgebra lineal
import matplotlib.pyplot as plt         # visualización gráfica: figuras, ejes, estilos

# Controles interactivos (sliders) para modificar parámetros dinámicamente en Jupyter
from ipywidgets import interact, FloatSlider   # widgets interactivos (sliders) para Jupyter

# Proyecto (requiere `pip install -e .` desde la raíz del repo).
# La lógica del modelo (ecuaciones, simulación, estado estacionario) vive en
# src/macroaicomp/models/islm.py. Importamos solo las 4 funciones que
# necesitamos: el notebook se limita a llamarlas, no reimplementa fórmulas.
import sys
sys.path.append('../../src')
from macroaicomp.models.islm import default_calibration, steady_state, simulate_shock, system_dynamics

## 1. El Marco Teórico: Ecuaciones y Parámetros

El modelo IS-LM dinámico en tiempo continuo describe una economía cerrada con las siguientes relaciones estructurales:

### 1.1 Ecuaciones de Comportamiento

1.  **Mercado Monetario (Curva LM):** El tipo de interés nominal ($i$) equilibra la oferta real de dinero con la demanda de saldos reales:
    $$M - P = \psi Y - \theta i$$
    Donde:
    *   $M$: Logaritmo de la oferta monetaria nominal (controlada por el Banco Central).
    *   $P$: Logaritmo del nivel de precios.
    *   $Y$: Nivel de producción real (renta).
    *   $\psi$: Sensibilidad de la demanda de dinero respecto a la renta.
    *   $\theta$: Sensibilidad de la demanda de dinero respecto al tipo de interés nominal.

    Al despejar el tipo de interés nominal ($i$), obtenemos el equilibrio del mercado monetario en función del estado de la economía ($Y, P$):
    $$i(Y, P) = \frac{P - M + \psi Y}{\theta}$$

2.  **Demanda Agregada (Curva IS):** La demanda agregada planeada ($Y^d$) depende negativamente del tipo de interés real ($r = i - \pi$):
    $$Y^d = \beta_0 - \beta_1(i - \pi)$$
    Donde:
    *   $\beta_0$: Demanda agregada autónoma (gasto público, consumo autónomo).
    *   $\beta_1$: Sensibilidad de la inversión y consumo respecto al tipo de interés real.
    *   $\pi = \frac{dP}{dt}$: Tasa de inflación observada en tiempo continuo.

3.  **Curva de Phillips (Dinámica de Precios):** La tasa de inflación responde a la presión de demanda (brecha de producción):
    $$\frac{dP}{dt} = \mu(Y - \bar{Y})$$
    Donde:
    *   $\bar{Y}$ (`ypot0`): Nivel de producto potencial o pleno empleo de largo plazo.
    *   $\mu$: Sensibilidad de la inflación respecto a la brecha de producción (ajuste de precios).

4.  **Ajuste del Producto (Dinámica de Producción):** La producción real ($Y$) se ajusta gradualmente hacia el nivel de demanda agregada:
    $$\frac{dY}{dt} = \nu(Y^d - Y)$$
    Donde:
    *   $\nu$: Velocidad de ajuste de la producción física frente al exceso de demanda.

---

### 1.2 Reducción a un Sistema Dinámico de Ecuaciones Diferenciales Ordinarias (ODEs)

Para simular la economía en el tiempo, reducimos las cuatro relaciones anteriores a un sistema de dos ecuaciones diferenciales ordinarias (ODEs) con dos variables de estado: la producción real ($Y$) y el nivel de precios ($P$).

Sustituyendo el tipo de interés nominal $i$ y la tasa de inflación $\frac{dP}{dt}$ en la demanda agregada $Y^d$, obtenemos:
$$Y^d = \beta_0 - \beta_1 \left( \frac{P - M + \psi Y}{\theta} - \mu(Y - \bar{Y}) \right)$$

Sustituyendo esta demanda agregada en la ecuación de ajuste de la producción, el sistema de dos variables de estado queda completamente determinado por:
$$\frac{dY}{dt} = \nu \left[ \beta_0 - \beta_1 \left( \frac{P - M + \psi Y}{\theta} - \mu(Y - \bar{Y}) \right) - Y \right]$$
$$\frac{dP}{dt} = \mu(Y - \bar{Y})$$

Este es el sistema continuo bidimensional que resolveremos numéricamente. Las variables $Y$ y $P$ se comportan como variables de estado lentas (continuas), mientras que $i$ y $Y^d$ son variables algebraicas que pueden dar saltos discretos e instantáneos cuando ocurre una política económica imprevista.


In [ ]:
# Esta celda solo FIJA NÚMEROS (Apéndice D del libro): todavía no calcula
# nada. default_calibration() devuelve un ISLMParameters, un dataclass
# (definido en src/macroaicomp/models/islm.py): una "ficha" con 8 casillas
# con nombre (theta, psi, beta1, mi, ni, beta0, m0, ypot0), cada una
# guardando un número. Usar default_calibration() en vez de escribir los
# números a mano evita errores de tecleo: los valores están centralizados en
# el código fuente, donde se testean con pytest antes de llegar al notebook.
# Al ejecutar veremos los 8 valores impresos con su descripción económica
# como comprobación visual de que no hay errores antes de usarlos en ningún
# cálculo.

# Cargar la calibración por defecto basada en los valores originales del libro.
# params es un objeto dataclass (ISLMParameters) que almacena la calibración de la economía.
params = default_calibration()

# Definir un diccionario didáctico con la descripción económica y el símbolo correspondiente
# a cada parámetro técnico del modelo para facilitar la comprensión del estudiante.
descriptions = {
    "theta": "Sensibilidad de la demanda de dinero al tipo de interés nominal [θ]",
    "psi": "Sensibilidad de la demanda de dinero a la renta real (PIB) [ψ]",
    "beta1": "Sensibilidad de la inversión y consumo al tipo de interés real [β1]",
    "mi": "Ajuste de precios ante brecha de producción (Curva de Phillips) [μ]",
    "ni": "Velocidad de ajuste de la producción física en mercado de bienes [ν]",
    "beta0": "Demanda agregada autónoma base (Gasto público + Consumo base) [β0]",
    "m0": "Oferta monetaria nominal fijada por el Banco Central [M0]",
    "ypot0": "Producción potencial de pleno empleo a largo plazo [Y_barra]"
}

# Imprimir de forma estructurada los parámetros con su explicación económica
print("CALIBRACIÓN ECONÓMICA DE REFERENCIA (Valores base del Libro):")
print("=" * 75)
print(f"{'Variable':<12} | {'Valor':<6} | {'Descripción Económica':<50}")
print("-" * 75)
for param_name, value in vars(params).items():
    desc = descriptions.get(param_name, "Parámetro del modelo")
    print(f"  {param_name:<10} | {value:<6} | {desc:<50}")
print("=" * 75)

## 2. Equilibrio de Largo Plazo (Estado Estacionario)

En el largo plazo, todas las variables dinámicas se estabilizan. Esto significa que las derivadas temporales son nulas:
$$\frac{dY}{dt} = 0 \quad \text{y} \quad \frac{dP}{dt} = 0$$

A partir de estas dos condiciones de equilibrio de largo plazo, podemos deducir analíticamente los valores de estado estacionario ($Y^*, P^*, i^*$):

1.  **De la Curva de Phillips:** $\frac{dP}{dt} = \mu(Y^* - \bar{Y}) = 0 \Rightarrow Y^* = \bar{Y}$.
    *   *Significado:* La producción de largo plazo coincide necesariamente con el **producto potencial** ($2000.0$).
2.  **Del Ajuste de Producción:** $\frac{dY}{dt} = \nu(Y^d - Y^*) = 0 \Rightarrow Y^d = Y^* = \bar{Y}$.
    *   *Significado:* La demanda agregada planeada iguala a la producción en el largo plazo.
3.  **De la Curva IS:** Dado que la tasa de inflación es nula en estado estacionario ($\pi^* = 0$), el tipo de interés real es igual al tipo de interés nominal ($r^* = i^*$). Por tanto:
    $$\bar{Y} = \beta_0 - \beta_1 i^* \Rightarrow i^* = \frac{\beta_0 - \bar{Y}}{\beta_1}$$
    *   *Cálculo:* $i^* = \frac{2100.0 - 2000.0}{50.0} = 2.0\%$ (0.02).
4.  **De la Curva LM:** Despejando el nivel de precios de equilibrio $P^*$:
    $$P^* = \theta i^* + M_0 - \psi \bar{Y}$$
    *   *Cálculo:* $P^* = 0.5 \cdot 2.0 + 100.0 - 0.01 \cdot 2000.0 = 1.0 + 100.0 - 20.0 = 81.0$.

A continuación, ejecutamos la función de cálculo numérico para confirmar que obtenemos los mismos resultados.


In [ ]:
# steady_state() es una FUNCIÓN: le pasamos los parámetros (params) y nos
# devuelve un diccionario con los valores de equilibrio de largo plazo
# (Y, P, i, Yd, dP, dY). Por dentro resuelve las condiciones dY/dt = 0,
# dP/dt = 0 usando las fórmulas analíticas derivadas en la celda de teoría
# de la Sección 2: Y* = ypot0 (producción potencial), i* = (beta0 - Y*)/beta1,
# P* = theta*i* + m0 - psi*Y*. No necesitamos saber CÓMO lo calcula para
# usarla: solo qué entra (parámetros) y qué sale (diccionario con 6 claves).
# Las cadenas con "f" delante (f-strings) permiten meter el valor de una
# variable dentro del texto escribiéndola entre llaves {}. El ":" dentro de
# las llaves controla el formato de impresión: .2f = 2 decimales fijos,
# .2% = porcentaje con 2 decimales (multiplica por 100 y añade el signo %),
# .4f = 4 decimales fijos. El valor real guardado en la variable no cambia.

# ==============================================================================
# CÁLCULO NUMÉRICO DEL ESTADO ESTACIONARIO ANALÍTICO
# ==============================================================================

# Calcular el estado estacionario inicial a partir de las fórmulas analíticas derivadas
# pasándole nuestro objeto de parámetros base.
ss = steady_state(params)

# Imprimir con detalle y formato adecuado para el alumno los valores resultantes.
# Al ejecutar veremos los 6 valores: Y*=2000, P*=81, i*=2%, Yd*=2000, dP=0, dY=0.
print("VALORES DE EQUILIBRIO DE LARGO PLAZO (ESTADO ESTACIONARIO BASE):")
print("-" * 65)
print(f"  Renta de pleno empleo (Y*) : {ss['Y']:.2f}")
print(f"  Nivel de precios (P*)      : {ss['P']:.2f} (Logaritmo de precios)")
print(f"  Tipo de interés (i*)       : {ss['i']:.2%} ({ss['i']:.4f} en tanto por uno)")
print(f"  Demanda agregada (Yd*)     : {ss['Yd']:.2f} (debe ser idéntica a Y*)")
print(f"  Tasa de inflación (dP/dt)  : {ss['dP']:.2f} (estabilidad absoluta de precios)")
print(f"  Ajuste del producto (dY/dt): {ss['dY']:.2f} (estabilidad de la producción)")
print("-" * 65)

In [ ]:
# Verificamos que el estado estacionario calculado numéricamente coincide con
# el oráculo del Apéndice D (MATLAB) recogido en oraculo.md.
# np.testing.assert_allclose compara dos valores y SOLO lanza un error si la
# diferencia supera la tolerancia atol. No usamos "==" porque la aritmética
# con decimales casi nunca da resultados exactamente iguales (errores de
# redondeo internos). Si el port a Python tuviera un error, esta celda
# lanzaría AssertionError y detendría la ejecución antes de seguir
# construyendo gráficos sobre un resultado incorrecto.
np.testing.assert_allclose(ss['Y'], 2000.0, atol=1e-6)
np.testing.assert_allclose(ss['P'], 81.0, atol=1e-6)
np.testing.assert_allclose(ss['i'], 2.0, atol=1e-6)
np.testing.assert_allclose(ss['Yd'], 2000.0, atol=1e-6)
np.testing.assert_allclose(ss['dP'], 0.0, atol=1e-6)
np.testing.assert_allclose(ss['dY'], 0.0, atol=1e-6)
print("OK: coincide con el oráculo MATLAB (Apéndice D).")

## 3. Detrás de la Escena: Resolución Numérica de ODEs en Python

Para simular una economía dinámica en tiempo continuo, debemos resolver el sistema de ecuaciones diferenciales:
$$\frac{d\mathbf{x}}{dt} = \mathbf{f}(t, \mathbf{x}, \mathbf{\theta})$$
Donde el vector de estado es $\mathbf{x}(t) = [Y(t), P(t)]^T$ y los parámetros son $\mathbf{\theta}$. En Python, utilizamos la función científica `scipy.integrate.solve_ivp`. 

### ¿Cómo funciona `solve_ivp`?
1. **El Algoritmo de Integración:** Por defecto, utiliza el método `'RK45'` (Runge-Kutta de orden 4 con estimación de error de orden 5). Este algoritmo evalúa el campo vectorial (las derivadas) en múltiples puntos de cada subintervalo temporal y ajusta el tamaño del paso dinámicamente para garantizar que la solución numérica no se desvíe de la trayectoria teórica.
2. **Definición de la función de derivadas:** Requiere que le proporcionemos una función que tome el tiempo actual `t` y el estado actual `state` y devuelva el vector de derivadas `[dY/dt, dP/dt]`. Aunque el tiempo no aparece explícitamente en el lado derecho de nuestras ecuaciones (es un sistema autónomo), `solve_ivp` requiere obligatoriamente que `t` sea el primer argumento de la función de derivadas.
3. **El Intervalo Temporal (`t_span`):** Define desde qué instante hasta qué instante simulamos la transición (por ejemplo, de $t=0$ a $t=30$ períodos).
4. **Condiciones Iniciales (`y0`):** La posición de partida del sistema en el instante $t=0$ (en este caso, los valores de producción y precios en el estado estacionario inicial pre-shock).

A continuación se muestra de forma explícita cómo programar la dinámica del modelo utilizando la biblioteca estándar y los parámetros cargados. De este modo puedes ver la lógica que opera dentro de la biblioteca `macroaicomp`:


In [ ]:
# Esta celda DEFINE (no ejecuta) la función custom_system_dynamics(), el
# "motor" del modelo IS-LM: dado un punto (Y, P) en el espacio de estados y
# unos parámetros, devuelve [dY/dt, dP/dt], es decir, hacia dónde se mueve
# la economía en ese instante (el "campo vectorial"). solve_ivp llamará a
# esta función miles de veces durante la integración numérica. La definimos
# aquí desglosada paso a paso para que veas exactamente qué ecuaciones hay
# dentro de simulate_shock() — en el paquete macroaicomp esta misma lógica
# está ya implementada en system_dynamics(), pero aquí la puedes leer, tocar
# y modificar sin salir del notebook.
# "def nombre(args):" define una FUNCIÓN: el código de dentro no se ejecuta
# al escribir la celda, solo cuando alguien llama a la función más abajo.

# ==============================================================================
# DEFINICIÓN DETALLADA Y COMENTADA DEL SISTEMA DINÁMICO IS-LM
# ==============================================================================

def custom_system_dynamics(t, state, params):
    """
    Define el sistema de ecuaciones diferenciales del modelo IS-LM para el integrador.
    
    Parámetros:
    -----------
    t : float
        Tiempo (requerido por scipy.integrate.solve_ivp, aunque el sistema es autónomo).
    state : array_like
        Vector de estado de tamaño 2: [Y, P]
        - Y: Producción real en el instante t.
        - P: Nivel de precios en el instante t.
    params : ISLMParameters
        Objeto con los parámetros estructurales calibrados.
        
    Devuelve:
    ---------
    ndarray
        Derivadas temporales [dY_dt, dP_dt] para que solve_ivp las integre.
    """
    # 1. Despaquetar el estado de la economía: producción real (Y) y nivel de precios (P)
    Y, P = state
    
    # 2. Curva de Phillips (Dinámica de Precios / Inflación)
    # dP_dt = mi * (Y - Y_potencial)
    # Representa cómo responde la inflación al exceso de demanda (brecha de producción).
    # Si Y > ypot0 (brecha de producción positiva), hay presión inflacionaria y los precios suben.
    # Si Y < ypot0 (recesión), hay presiones deflacionarias y los precios bajan.
    dP_dt = params.mi * (Y - params.ypot0)
    
    # 3. Mercado Monetario / Curva LM (Equilibrio de tipos de interés nominales)
    # i = (P - M + psi * Y) / theta
    # Esta ecuación se obtiene al igualar la oferta monetaria real (M - P) con la demanda
    # real de saldos monetarios L(Y, i) = psi * Y - theta * i, y despejar el tipo nominal i.
    i = (P - params.m0 + params.psi * Y) / params.theta
    
    # 4. Mercado de Bienes / Demanda Agregada Planeada (Curva IS)
    # Yd = beta0 - beta1 * r
    # Donde r es el tipo de interés real. En tiempo continuo, el tipo de interés real
    # es r = i - dP_dt (el tipo de interés nominal ajustado por la inflación observada dP_dt).
    r = i - dP_dt
    Yd = params.beta0 - params.beta1 * r
    
    # 5. Ajuste gradual de la producción real (Ajuste del mercado de bienes)
    # dY_dt = ni * (Yd - Y)
    # La producción física (Y) no cambia instantáneamente. Las empresas tardan un tiempo
    # en reaccionar y ajustar la producción física ante un exceso de demanda (Yd - Y).
    # La velocidad con la que se ajusta el mercado viene determinada por el parámetro 'ni'.
    dY_dt = params.ni * (Yd - Y)
    
    # Devolver las derivadas como un vector columna para que solve_ivp las integre
    return np.array([dY_dt, dP_dt])

## 4. Transición Dinámica y Shocks de Política (Simulación Interactiva)

### 4.1 El Mecanismo de Transmisión Económica
Cuando se produce un **shock monetario expansivo** ($M_0 \uparrow$):
1.  **Impacto inmediato (Corto Plazo):** Los precios ($P$) son rígidos en el instante inicial, por lo que la oferta de dinero real ($M - P$) se expande. Esto desplaza la curva LM hacia abajo, haciendo que el tipo de interés ($i$) caiga de forma abrupta.
2.  **Efecto de Demanda:** La caída en el coste del capital estimula la demanda agregada ($Y^d > Y$) a través de la curva IS. La producción ($Y$) empieza a crecer gradualmente guiada por la velocidad de ajuste $\nu$.
3.  **Mecanismo de Ajuste a Medio Plazo:** A medida que $Y$ supera la renta de pleno empleo $\bar{Y}$, aparece una brecha de producción positiva, lo que presiona los precios al alza (inflación positiva por Curva de Phillips).
4.  **Retorno al Largo Plazo (Neutralidad):** El aumento continuo de precios reduce los saldos monetarios reales ($M - P \downarrow$), desplazando la curva LM de nuevo hacia arriba. El tipo de interés sube y el producto vuelve a contraerse paulatinamente hasta retornar a la capacidad potencial $\bar{Y}$.

### 4.2 El Diagrama de Fases en el Plano $(Y, P)$
Para analizar geométricamente la estabilidad y transición, construimos el diagrama de fases localizando las curvas de demarcación donde las derivadas son nulas:
*   **Locus $\dot{P} = 0$:** Ocurre cuando $Y = \bar{Y}$. Es una línea vertical en el nivel de producto potencial. A la derecha, los precios suben ($\dot{P} > 0$); a la izquierda, bajan ($\dot{P} < 0$).
*   **Locus $\dot{Y} = 0$:** Ocurre cuando la demanda es igual a la producción ($Y^d = Y$). Sustituyendo las ecuacionesLM y Phillips obtenemos una relación lineal decreciente en el plano $(Y, P)$:
    $$P = M + \theta \left( \frac{\beta_0}{\beta_1} - \mu \bar{Y} \right) + \left( \theta \mu - \frac{\theta}{\beta_1} - \psi \right) Y$$
    Por debajo de esta curva, la producción se expande ($\dot{Y} > 0$); por encima, se contrae ($\dot{Y} < 0$).

Utiliza los deslizadores a continuación para simular este comportamiento dinámico de forma interactiva:


In [ ]:
# "def nombre(args):" define una FUNCIÓN reutilizable: no se ejecuta al
# escribirla, solo cuando interact() la llama más abajo. plot_shock_response()
# simula la economía IS-LM con los valores de m0 y beta0 que elijas en los
# sliders y dibuja 3 paneles: (1) evolución temporal de Y, (2) evolución
# temporal de P, (3) diagrama de fases en el plano (Y, P) con el campo
# vectorial, los locus dY/dt=0 y dP/dt=0, y la trayectoria simulada.
# simulate_shock() llama internamente a solve_ivp (método RK45) para
# integrar las ODEs. Al mover los sliders verás en vivo cómo cambian las
# trayectorias — no hay que re-ejecutar la celda.
# interact() conecta la función a los FloatSlider: cada vez que mueves uno,
# ipywidgets vuelve a llamar a plot_shock_response() con los nuevos valores
# y redibuja los 3 paneles. El nombre del argumento (m0_shock=...) tiene que
# coincidir exactamente con el de la definición de la función. El ";" al
# final de interact(...) evita que Jupyter imprima también la representación
# de texto del widget debajo de los gráficos.

# ==============================================================================
# SIMULACIÓN INTERACTIVA DE SHOCKS Y REPRESENTACIÓN EN 3 PANELES (CON DIAGRAMA DE FASES)
# ==============================================================================

def plot_shock_response(m0_shock=110.0, beta0_shock=2100.0):
    """
    Simula la respuesta dinámica ante shocks en la oferta monetaria y el gasto autónomo.
    Muestra gráficamente las trayectorias de transición temporal y el diagrama de fases.
    """
    # 1. Cargar la calibración por defecto del modelo (situación inicial antes de shocks)
    params_sim = default_calibration()
    
    # 2. Calcular analíticamente el estado estacionario inicial para usarlo como condición inicial
    ss_init = steady_state(params_sim)
    initial_state = np.array([ss_init['Y'], ss_init['P']])
    
    # 3. Aplicar los nuevos parámetros del shock seleccionados por el alumno en los deslizadores
    params_sim.m0 = m0_shock
    params_sim.beta0 = beta0_shock
    
    # 4. Resolver numéricamente las ecuaciones diferenciales del sistema con solve_ivp
    # Definimos el rango de tiempo de simulación (de t=0 a t=30 períodos)
    t_span = (0, 30)
    # Creamos un mallado de evaluación temporal para guardar 300 puntos y obtener curvas suaves
    t_eval = np.linspace(0, 30, 300)
    
    # Llamamos al simulador del modelo (que ejecuta solve_ivp con RK45)
    res = simulate_shock(params_sim, initial_state, t_span, t_eval)
    
    # Extraer las trayectorias temporales resueltas de la producción (Y) y los precios (P)
    Y_path = res.y[0]
    P_path = res.y[1]
    
    # 5. Calcular el nuevo estado estacionario a largo plazo para representarlo como referencia final
    ss_final = steady_state(params_sim)
    
    # 6. Configurar la figura con 3 paneles para mostrar Renta, Precios y Diagrama de Fases
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    
    # --- PANEL 1: DINÁMICA DE LA RENTA (Y) EN EL TIEMPO ---
    # Dibujar la evolución del PIB simulado
    axs[0].plot(res.t, Y_path, color='#004C97', linewidth=2.5, label='Producción (Y)')
    # Línea horizontal discontinua de la producción potencial
    axs[0].axhline(params_sim.ypot0, color='red', linestyle='--', alpha=0.7, label='Renta Potencial')
    axs[0].set_title('Evolución Temporal de la Renta (Y)', fontsize=11, fontweight='bold', pad=10)
    axs[0].set_xlabel('Tiempo (t)', fontsize=9)
    axs[0].set_ylabel('Producción (Y)', fontsize=9)
    axs[0].grid(True, linestyle=':', alpha=0.6)
    axs[0].legend(loc='lower right', fontsize=8)
    
    # --- PANEL 2: DINÁMICA DEL NIVEL DE PRECIOS (P) EN EL TIEMPO ---
    # Dibujar la evolución del nivel de precios simulado
    axs[1].plot(res.t, P_path, color='#8EAD3A', linewidth=2.5, label='Precios (P)')
    # Línea horizontal del nivel de precios antes del shock
    axs[1].axhline(ss_init['P'], color='gray', linestyle=':', alpha=0.5, label='Precios Pre-shock')
    # Línea horizontal del nuevo equilibrio de precios
    axs[1].axhline(ss_final['P'], color='black', linestyle='--', alpha=0.7, label='Nuevo E.E. Largo Plazo')
    axs[1].set_title('Evolución Temporal de Precios (P)', fontsize=11, fontweight='bold', pad=10)
    axs[1].set_xlabel('Tiempo (t)', fontsize=9)
    axs[1].set_ylabel('Nivel de Precios (P)', fontsize=9)
    axs[1].grid(True, linestyle=':', alpha=0.6)
    axs[1].legend(loc='lower right', fontsize=8)
    
    # --- PANEL 3: DIAGRAMA DE FASES EN EL PLANO (Y, P) ---
    # Dibujar la trayectoria dinámica completa en el espacio de estados
    axs[2].plot(Y_path, P_path, color='#7A3E9F', linewidth=3, label='Trayectoria dinámica')
    
    # Dibujar el locus de equilibrio de precios dP/dt = 0 (Línea vertical en ypot0)
    axs[2].axvline(params_sim.ypot0, color='#D95319', linestyle='--', linewidth=2, label=r'$\dot{P} = 0$ (Pleno Empleo)')
    
    # Dibujar el locus de equilibrio de la demanda agregada dY/dt = 0
    # Definimos el rango del eje de producción en función de los valores simulados
    Y_vals = np.linspace(min(Y_path) - 15, max(Y_path) + 15, 100)
    # Fórmulas de la pendiente e intersección para los locus
    slope = params_sim.theta * params_sim.mi - params_sim.theta / params_sim.beta1 - params_sim.psi
    intercept_init = ss_init['P'] - slope * params_sim.ypot0
    intercept_final = ss_final['P'] - slope * params_sim.ypot0
    
    P_locus_init = intercept_init + slope * Y_vals
    P_locus_final = intercept_final + slope * Y_vals
    
    # Dibujar las curvas LM/IS de equilibrio para el estado inicial y final
    axs[2].plot(Y_vals, P_locus_init, color='#0072BD', linestyle=':', alpha=0.5, label=r'$\dot{Y} = 0$ (Inicial)')
    axs[2].plot(Y_vals, P_locus_final, color='#0072BD', linestyle='--', linewidth=2, label=r'$\dot{Y} = 0$ (Final)')
    
    # Dibujar el campo vectorial de derivadas en segundo plano mediante flechas (quiver)
    # Esto ayuda a visualizar las fuerzas de atracción/repulsión del sistema dinámico.
    Y_grid, P_grid = np.meshgrid(
        np.linspace(min(Y_path) - 8, max(Y_path) + 8, 12), 
        np.linspace(min(P_path) - 0.8, max(P_path) + 0.8, 12)
    )
    dY_grid = np.zeros_like(Y_grid)
    dP_grid = np.zeros_like(P_grid)
    
    # Evaluar las derivadas en cada intersección de la rejilla
    for r in range(Y_grid.shape[0]):
        for c in range(Y_grid.shape[1]):
            derivs = system_dynamics(0.0, np.array([Y_grid[r, c], P_grid[r, c]]), params_sim)
            dY_grid[r, c] = derivs[0]
            dP_grid[r, c] = derivs[1]
    
    # Normalizar la longitud de las flechas para que se vean uniformes y legibles en el gráfico
    norm = np.hypot(dY_grid, dP_grid)
    norm[norm == 0] = 1.0
    dY_grid /= norm
    dP_grid /= norm
    
    # Dibujar las flechas del campo vectorial
    axs[2].quiver(Y_grid, P_grid, dY_grid, dP_grid, color='lightgray', alpha=0.6, scale=25, width=0.003)
    
    # Dibujar los marcadores del punto de partida y del destino final del sistema
    axs[2].scatter(ss_init['Y'], ss_init['P'], color='gray', s=100, zorder=5, label='E.E. Inicial')
    axs[2].scatter(ss_final['Y'], ss_final['P'], color='black', marker='*', s=200, zorder=5, label='E.E. Final')
    
    # Añadir una flecha direccional en el centro de la trayectoria para marcar el sentido temporal del movimiento
    half_idx = len(Y_path) // 2
    axs[2].annotate('', xy=(Y_path[half_idx], P_path[half_idx]), 
                     xytext=(Y_path[half_idx-1], P_path[half_idx-1]),
                     arrowprops=dict(arrowstyle="->", color='#7A3E9F', lw=3, mutation_scale=15))
    
    # Ajustes estéticos finales del tercer panel
    axs[2].set_title('Diagrama de Fases en el Plano (Y, P)', fontsize=11, fontweight='bold', pad=10)
    axs[2].set_xlabel('Producción (Y)', fontsize=9)
    axs[2].set_ylabel('Nivel de Precios (P)', fontsize=9)
    axs[2].set_xlim(min(Y_path) - 10, max(Y_path) + 10)
    axs[2].set_ylim(min(P_path) - 1, max(P_path) + 1)
    axs[2].grid(True, linestyle=':', alpha=0.6)
    axs[2].legend(loc='best', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Crear los controles interactivos de ipywidgets en el cuaderno Jupyter.
# m0_shock por defecto simula un shock expansivo monetario del 10% (de 100 a 110).
# beta0_shock por defecto simula gasto público estable en 2100.
interact(
    plot_shock_response, 
    m0_shock=FloatSlider(value=110.0, min=80.0, max=120.0, step=1.0, description='Oferta Monetaria (M0)'),
    beta0_shock=FloatSlider(value=2100.0, min=1800.0, max=2400.0, step=10.0, description='Gasto Autónomo (B0)')
);

## 5. Verificación frente al oráculo

Comparamos contra los valores reportados en el libro y reproducidos por el
código MATLAB del Apéndice D (`referencia/islm.m`), recogidos en
`oraculo.md`:

| Magnitud | Valor esperado (oráculo) |
|---|---|
| Estado estacionario inicial — $Y$ | 2000.0 |
| Estado estacionario inicial — $P$ | 81.0 |
| Estado estacionario inicial — $i$ | 2.0 |
| Estado estacionario inicial — $Y^d$ | 2000.0 |
| Estado estacionario inicial — $\Delta P$ | 0.0 |
| Estado estacionario inicial — $\Delta Y$ | 0.0 |
| Nuevo SS tras shock monetario $M_0: 100 \to 101$ — $Y$ | 2000.0 |
| Nuevo SS tras shock monetario $M_0: 100 \to 101$ — $P$ | 82.0 |
| Nuevo SS tras shock monetario $M_0: 100 \to 101$ — $i$ | 2.0 |

Así puedes comparar a simple vista, sin abrir `oraculo.md`, el número que
debería salir en cada celda siguiente con el que realmente sale.

## 6. Buenas Prácticas Aplicadas en este Laboratorio

Fíjate en las siguientes decisiones de diseño técnico tomadas para estructurar esta práctica de forma ejemplar:
1.  **Código Modular**: La lógica matemática de las ecuaciones del modelo y el solucionador numérico no están en este notebook. Están aislados en el archivo modular `src/macroaicomp/models/islm.py` para asegurar su reutilización.
2.  **Calibración Aislada**: No hay números "mágicos" embebidos en el cálculo. Los parámetros se cargan desde un diccionario o estructura al principio.
3.  **Independencia Didáctica**: El notebook funciona de manera autónoma como un *frontend* interactivo sin saturar al alumno con detalles de codificación de bajo nivel.


## 7. Cuaderno de Bitácora (Actividades para el Alumno)

Responde en tu Cuaderno de Bitácora digital las siguientes preguntas basándote en tus observaciones de la simulación:

1.  **El Shock Monetario:** Deja el *Gasto Autónomo* ($B_0$) en su nivel base ($2100.0$) e incrementa la *Oferta Monetaria* ($M_0$) a $115.0$.
    *   ¿Qué ocurre con la Renta ($Y$) a corto plazo? ¿Por qué?
    *   ¿Qué ocurre con el nivel de precios ($P$) a largo plazo? Explica la relación porcentual entre el aumento de dinero y el aumento de precios en el equilibrio final.
    *   ¿Se cumple el principio de *neutralidad del dinero*? Justifica tu respuesta.
2.  **El Shock Fiscal:** Restaura $M_0$ a $100.0$ e incrementa el *Gasto Autónomo* a $2200.0$ (shock de política fiscal expansiva).
    *   Describe la trayectoria de la Renta ($Y$). ¿Aumenta de forma permanente o transitoria?
    *   ¿Qué ocurre con el tipo de interés nominal ($i$) a largo plazo? Explica el fenómeno económico conocido como *efecto expulsión* (crowding-out) a partir de los resultados y el gráfico.
3.  **Análisis del Diagrama de Fases (Espacio de Estados):**
    *   Observa la trayectoria de transición en el plano $(Y, P)$ en el Panel 3. ¿Por qué la trayectoria se inicia con un movimiento puramente horizontal hacia la derecha y no en diagonal desde el principio? Relaciona esto con el supuesto de rigidez instantánea de los precios ($P$) y el ajuste gradual de la renta ($Y$) ante excesos de demanda.
    *   Explica por qué la trayectoria cruza el locus $\dot{Y}=0$ con una tangente vertical. ¿Qué valor toma la derivada $\dot{Y}$ en ese instante exacto de cruce?
    *   ¿Cómo cambia la trayectoria en el diagrama si la velocidad de ajuste del mercado de bienes ($\nu$) disminuye drásticamente a $0.05$? Compárala con el comportamiento bajo la calibración original ($\nu=0.2$).
